# Crop Yield Prediction — Model Training
Trains a **GradientBoostingRegressor** (numeric yield in kg/ha) and derives
Low / Medium / High categories using per-crop percentile thresholds.

Inputs: `crop`, `soil_temp`, `nitrogen`, `phosphorus`, `potassium`, `moisture`, `humidity`, `air_temp`
Output: `yield_kg_ha` (numeric) + `yield_category` (Low / Medium / High)

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 3000

CROP_PARAMS = {
    'rice':      dict(st=(25,35), n=(80,120),  p=(40,60),  k=(40,60),  mo=(60,85), hu=(75,95), at=(22,32), by=4500),
    'wheat':     dict(st=(15,25), n=(100,140), p=(50,80),  k=(40,60),  mo=(40,65), hu=(50,75), at=(10,24), by=3200),
    'maize':     dict(st=(20,30), n=(120,160), p=(60,90),  k=(60,90),  mo=(50,75), hu=(55,80), at=(18,28), by=5500),
    'sugarcane': dict(st=(25,35), n=(100,150), p=(50,80),  k=(80,120), mo=(65,85), hu=(70,90), at=(24,34), by=70000),
    'cotton':    dict(st=(25,35), n=(100,140), p=(50,80),  k=(40,60),  mo=(45,70), hu=(50,75), at=(22,32), by=1800),
    'jute':      dict(st=(25,35), n=(60,100),  p=(30,60),  k=(30,50),  mo=(70,90), hu=(75,95), at=(24,34), by=2500),
    'coffee':    dict(st=(20,28), n=(80,120),  p=(40,70),  k=(60,100), mo=(55,75), hu=(70,90), at=(18,26), by=900),
    'tea':       dict(st=(18,26), n=(80,120),  p=(30,60),  k=(40,80),  mo=(60,80), hu=(75,90), at=(15,24), by=1200),
    'rubber':    dict(st=(25,32), n=(60,100),  p=(30,60),  k=(60,100), mo=(65,85), hu=(75,90), at=(22,30), by=1500),
    'soybean':   dict(st=(20,30), n=(20,60),   p=(60,90),  k=(40,70),  mo=(50,70), hu=(60,80), at=(18,28), by=2800),
    'groundnut': dict(st=(22,32), n=(20,50),   p=(40,70),  k=(30,60),  mo=(45,70), hu=(55,75), at=(20,30), by=2200),
    'sunflower': dict(st=(20,30), n=(80,120),  p=(50,80),  k=(40,70),  mo=(40,65), hu=(45,70), at=(18,28), by=1900),
    'tomato':    dict(st=(20,28), n=(100,150), p=(60,90),  k=(80,120), mo=(55,75), hu=(60,80), at=(18,28), by=35000),
    'onion':     dict(st=(18,26), n=(80,120),  p=(40,70),  k=(60,100), mo=(50,70), hu=(55,75), at=(15,25), by=20000),
    'potato':    dict(st=(15,22), n=(100,150), p=(60,100), k=(80,120), mo=(60,80), hu=(60,80), at=(12,22), by=22000),
}

rows = []
per_crop = n // len(CROP_PARAMS)

for crop, p in CROP_PARAMS.items():
    for _ in range(per_crop):
        soil_temp  = rng.uniform(*p['st'])
        nitrogen   = rng.uniform(*p['n'])
        phosphorus = rng.uniform(*p['p'])
        potassium  = rng.uniform(*p['k'])
        moisture   = rng.uniform(*p['mo'])
        humidity   = rng.uniform(*p['hu'])
        air_temp   = rng.uniform(*p['at'])

        def norm(val, lo, hi): return (val - lo) / (hi - lo)
        factor = (
            0.85
            + 0.05 * norm(nitrogen,   *p['n'])
            + 0.04 * norm(phosphorus, *p['p'])
            + 0.03 * norm(potassium,  *p['k'])
            + 0.04 * norm(moisture,   *p['mo'])
            + 0.02 * norm(humidity,   *p['hu'])
            + 0.02 * norm(soil_temp,  *p['st'])
            - 0.01 * abs(norm(air_temp, *p['at']) - 0.5)
        )
        yield_kg = round(p['by'] * factor * rng.normal(1.0, 0.06), 1)
        rows.append({
            'crop': crop, 'soil_temp': round(soil_temp, 2),
            'nitrogen': round(nitrogen, 2), 'phosphorus': round(phosphorus, 2),
            'potassium': round(potassium, 2), 'moisture': round(moisture, 2),
            'humidity': round(humidity, 2), 'air_temp': round(air_temp, 2),
            'yield_kg_ha': max(0, yield_kg),
        })

df = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)
print(df.shape)
df.head()

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import pickle

crop_le = LabelEncoder()
df['crop_enc'] = crop_le.fit_transform(df['crop'])

FEATURES = ['crop_enc', 'soil_temp', 'nitrogen', 'phosphorus',
            'potassium', 'moisture', 'humidity', 'air_temp']
X = df[FEATURES]
y = df['yield_kg_ha']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

model = GradientBoostingRegressor(n_estimators=300, max_depth=5, learning_rate=0.08, random_state=42)
model.fit(X_tr, y_tr)

preds = model.predict(X_te)
print(f'MAE : {mean_absolute_error(y_te, preds):.2f} kg/ha')
print(f'R²  : {r2_score(y_te, preds):.4f}')

In [ ]:
# Per-crop percentile thresholds for Low/Medium/High category
thresholds = {}
for crop, grp in df.groupby('crop'):
    thresholds[crop] = {
        'low_max':  grp['yield_kg_ha'].quantile(0.33),
        'high_min': grp['yield_kg_ha'].quantile(0.67),
    }

# Save all artifacts
with open('yield_regressor.pkl',  'wb') as f: pickle.dump(model,      f)
with open('yield_crop_le.pkl',    'wb') as f: pickle.dump(crop_le,    f)
with open('yield_thresholds.pkl', 'wb') as f: pickle.dump(thresholds, f)

print('✅ Saved: yield_regressor.pkl, yield_crop_le.pkl, yield_thresholds.pkl')
print('Crops supported:', list(crop_le.classes_))